Piping a Prompt, Model, and an Output Parser

In [2]:
import os
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

In [5]:
list_instructions = CommaSeparatedListOutputParser().get_format_instructions()

In [6]:
list_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [7]:
chat_template = ChatPromptTemplate.from_messages([('human',
                                                   "I've recently adopted a {pet}. Could you suggest three {pet} names? \n" + list_instructions)])

In [8]:
print(chat_template.messages[0].prompt.template)

I've recently adopted a {pet}. Could you suggest three {pet} names? 
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [9]:
chat = ChatGroq(model_name="llama-3.1-8b-instant",
                model_kwargs= {'seed':365},
                temperature = 0,
                max_tokens= 100)

In [10]:
list_output_parser = CommaSeparatedListOutputParser()

In [11]:
chat_template_result = chat_template.invoke({'pet':'dog'})

In [12]:
chat_result = chat.invoke(chat_template_result)

In [13]:
list_output_parser.invoke(chat_result)

['Congratulations on your new furry family member. Here are three dog name suggestions: ',
 'Max',
 'Rufus',
 'Buddy']

In [14]:
chain = chat_template | chat | list_output_parser

In [15]:
chain.invoke({'pet':'dog'})

['Congratulations on your new furry family member. Here are three dog name suggestions: ',
 'Max',
 'Rufus',
 'Buddy']

Batching

In [16]:
chat_template = ChatPromptTemplate.from_messages([('human',
                                                   "I've recently adopted a {pet} which is a {breed}. Could you suggest several training tips?")])

In [17]:
chain = chat_template | chat

In [18]:
chain.invoke({'pet':'dog', 'breed':'shepherd'})

AIMessage(content='Congratulations on adopting a shepherd dog. Shepherds are highly intelligent and energetic breeds that require consistent training and socialization. Here are some training tips to help you get started:\n\n1. **Establish a routine**: Shepherds thrive on routine, so create a schedule for feeding, exercise, and training. This will help them feel secure and develop good habits.\n2. **Use positive reinforcement**: Reward your dog with treats, praise, and affection when they perform a desired behavior. Avoid punishment or negative', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 53, 'total_tokens': 153, 'completion_time': 0.110695085, 'completion_tokens_details': None, 'prompt_time': 0.003090442, 'prompt_tokens_details': None, 'queue_time': 0.005630551, 'total_time': 0.113785527}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'length', 'logprob

In [20]:
%%time
chain.batch([{'pet':'dog', 'breed':'shepherd'},
             {'pet':'cat', 'breed':'persian'}])

CPU times: total: 31.2 ms
Wall time: 420 ms


[AIMessage(content='Congratulations on adopting a shepherd dog. Shepherds are highly intelligent and energetic breeds that require consistent training and socialization. Here are some training tips to help you get started:\n\n1. **Establish a routine**: Shepherds thrive on routine, so create a schedule for feeding, exercise, and training. This will help them feel secure and develop good habits.\n2. **Use positive reinforcement**: Reward your dog with treats, praise, and affection when they perform a desired behavior. Avoid punishment or negative', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 53, 'total_tokens': 153, 'completion_time': 0.105817699, 'completion_tokens_details': None, 'prompt_time': 0.003543307, 'prompt_tokens_details': None, 'queue_time': 0.005437829, 'total_time': 0.109361006}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'length', 'logpro

In [21]:
%%time
chain.invoke({'pet':'dog', 'breed':'shepherd'})

CPU times: total: 15.6 ms
Wall time: 761 ms


AIMessage(content='Congratulations on adopting a shepherd dog. Shepherds are highly intelligent and energetic breeds that require consistent training and socialization. Here are some training tips to help you get started:\n\n1. **Establish a routine**: Shepherds thrive on routine, so create a schedule for feeding, exercise, and training. This will help them feel secure and develop good habits.\n2. **Use positive reinforcement**: Reward your dog with treats, praise, and affection when they perform a desired behavior. Avoid punishment or negative', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 53, 'total_tokens': 153, 'completion_time': 0.104373616, 'completion_tokens_details': None, 'prompt_time': 0.003180332, 'prompt_tokens_details': None, 'queue_time': 0.005578113, 'total_time': 0.107553948}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'length', 'logprob

In [23]:
%%time
chain.invoke({'pet':'cat', 'breed':'persian'})

CPU times: total: 31.2 ms
Wall time: 546 ms


AIMessage(content='Congratulations on adopting a beautiful Persian cat. Persian cats can be a bit more challenging to train due to their independent nature and thick coats, but with patience and consistency, you can help your cat become a well-behaved and loving companion. Here are some training tips for your Persian cat:\n\n1. **Establish a routine**: Persian cats thrive on routine, so create a schedule for feeding, playtime, and sleep. This will help your cat feel secure and develop a sense of trust.\n2.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 54, 'total_tokens': 154, 'completion_time': 0.11890618, 'completion_tokens_details': None, 'prompt_time': 0.007964874, 'prompt_tokens_details': None, 'queue_time': 0.005633597, 'total_time': 0.126871054}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'length', 'logprobs': None, 'model_provider': 'groq'}, id=

Stream() Method:

In [34]:
chat = ChatGroq(model_name="llama-3.1-8b-instant",
                model_kwargs= {'seed':365},
                temperature = 0,
                max_tokens= 100 )

In [35]:
chain = chat_template | chat

In [36]:
response = chain.stream({'pet':'cat', 'breed':'persian'})

In [37]:
next(response)

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'groq'}, id='lc_run--019fc176-7b16-7731-9008-f85e9b7732f9', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

In [38]:
for i in response:
    print(i.content, end = "")

Congratulations on adopting a beautiful Persian cat. Persian cats can be a bit more challenging to train due to their independent nature and thick coats, but with patience and consistency, you can help your cat become a well-behaved and loving companion. Here are some training tips for your Persian cat:

1. **Establish a routine**: Persian cats thrive on routine, so create a schedule for feeding, playtime, and sleep. This will help your cat feel secure and develop a sense of trust.
2.